In [1]:
import os, random, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from mambapy.vim import VMamba, MambaConfig
from thop import profile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
class VimEncoder(nn.Module):
    def __init__(self, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        config = MambaConfig(d_model=d_model, n_layers=n_layers, d_state=d_state,
                              bidirectional=True, divide_output=True, pscan=True, use_cuda=False)
        self.encoder = VMamba(config)
        self.final_norm = nn.LayerNorm(d_model)
    def forward(self, tokens):
        return self.final_norm(self.encoder(tokens))

In [3]:
class WholeBrainPatchEmbed3D(nn.Module):
    def __init__(self, brain_size=256, patch_size=8, d_model=32):
        super().__init__()
        self.patch_size = patch_size
        self.grid_size = brain_size // patch_size
        self.n_tokens = self.grid_size ** 3
        self.d_model = d_model
        self.patch_conv = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.depth_embed = nn.Embedding(self.grid_size, d_model)
        self.height_embed = nn.Embedding(self.grid_size, d_model)
        self.width_embed = nn.Embedding(self.grid_size, d_model)
        with torch.no_grad():
            for emb in [self.depth_embed, self.height_embed, self.width_embed]:
                emb.weight.mul_(0.02)
        d, h, w = torch.meshgrid(torch.arange(self.grid_size), torch.arange(self.grid_size),
                                  torch.arange(self.grid_size), indexing="ij")
        self.register_buffer("coordinates", torch.stack([d, h, w], dim=-1).reshape(-1, 3), persistent=False)

    def forward(self, volume):
        tokens = self.patch_conv(volume).flatten(2).transpose(1, 2)
        coords = self.coordinates
        spatial = self.depth_embed(coords[:, 0]) + self.height_embed(coords[:, 1]) + self.width_embed(coords[:, 2])
        tokens = tokens + spatial[None, :, :]
        occupancy = F.max_pool3d((volume.abs() > 1e-6).float(), kernel_size=self.patch_size, stride=self.patch_size)
        valid = occupancy.flatten(1).bool()
        tokens = tokens * valid.unsqueeze(-1).to(tokens.dtype)
        return tokens, valid


class WholeBrainBranch(nn.Module):
    """One modality's whole-brain pipeline -- used standalone (single
    modality) or twice (once per modality) inside the multimodal model."""
    def __init__(self, brain_size=256, patch_size=8, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        self.patch_embed = WholeBrainPatchEmbed3D(brain_size, patch_size, d_model)
        self.vim = VimEncoder(d_model, n_layers, d_state)

    def forward(self, volume):
        tokens, valid = self.patch_embed(volume)
        tokens = self.vim(tokens)
        w = valid.unsqueeze(-1).to(tokens.dtype)
        pooled = (tokens * w).sum(dim=1) / w.sum(dim=1).clamp_min(1.0)
        return pooled


class WholeBrainModel(nn.Module):
    """Single-modality -- use for MRI-only or PET-only."""
    def __init__(self, brain_size=256, patch_size=8, d_model=32, n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.branch = WholeBrainBranch(brain_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, volume):
        pooled = self.branch(volume)
        return self.classifier(self.dropout(pooled))


class MultimodalWholeBrainModel(nn.Module):
    """Late fusion -- separate MRI/PET whole-brain branches, concatenated
    before the classifier."""
    def __init__(self, brain_size=256, patch_size=8, d_model=32, n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.mri_branch = WholeBrainBranch(brain_size, patch_size, d_model, n_layers, d_state)
        self.pet_branch = WholeBrainBranch(brain_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model * 2, n_classes)

    def forward(self, mri_vol, pet_vol):
        mri_pooled = self.mri_branch(mri_vol)
        pet_pooled = self.pet_branch(pet_vol)
        fused = torch.cat([mri_pooled, pet_pooled], dim=1)
        return self.classifier(self.dropout(fused))

In [4]:
COHORT_CSV       = "D:/mamba_model/thesis_cohort_final.csv"
WB_MRI_CACHE_AUG = "D:/mamba_model/preprocessed_cache_wholebrain_native_mri_aug"
WB_PET_CACHE_AUG = "D:/mamba_model/preprocessed_cache_wholebrain_native_pet_aug"
CKPT_DIR         = "D:/mamba_model/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

BRAIN_SIZE = 256  # no downsampling

df = pd.read_csv(COHORT_CSV)
sessions, labels = df["mri_session"].values, df["outcome_label"].values
X_tv, X_test, y_tv, y_test = train_test_split(sessions, labels, test_size=0.2, random_state=42, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv)
session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 126 | Val: 42 | Test: 42


In [5]:
COHORT_CSV       = "D:/mamba_model/thesis_cohort_final.csv"
WB_MRI_CACHE_AUG = "D:/mamba_model/preprocessed_cache_wholebrain_native_mri_aug"
WB_PET_CACHE_AUG = "D:/mamba_model/preprocessed_cache_wholebrain_native_pet_aug"
CKPT_DIR         = "D:/mamba_model/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

BRAIN_SIZE = 256  # native FastSurfer conformed resolution -- no downsampling

df = pd.read_csv(COHORT_CSV)
sessions, labels = df["mri_session"].values, df["outcome_label"].values
X_tv, X_test, y_tv, y_test = train_test_split(sessions, labels, test_size=0.2, random_state=42, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv)
session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 126 | Val: 42 | Test: 42


In [6]:
class WholeBrainDataset(Dataset):
    """Single-modality (MRI or PET), native resolution, no ROI cropping/masking."""
    def __init__(self, sessions, labels, cache_dir, is_mri=True, is_train=False):
        self.samples, self.cache_dir = [], cache_dir
        for session_id, label in zip(sessions, labels):
            key = session_id if is_mri else session_to_subject[session_id]
            self.samples.append((key, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((key, label, f"aug{seed}"))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        key, label, version = self.samples[idx]
        vol = np.array(np.load(f"{self.cache_dir}/{key}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        return torch.from_numpy(vol).unsqueeze(0), torch.tensor(label, dtype=torch.long), key


class MultimodalWholeBrainDataset(Dataset):
    """Pairs MRI and PET whole-brain volumes for the same subject/seed."""
    def __init__(self, sessions, labels, mri_cache_dir, pet_cache_dir, is_train=False):
        self.samples = []
        self.mri_cache_dir, self.pet_cache_dir = mri_cache_dir, pet_cache_dir
        for session_id, label in zip(sessions, labels):
            subject_id = session_to_subject[session_id]
            self.samples.append((session_id, subject_id, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((session_id, subject_id, label, f"aug{seed}"))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        mri_key, pet_key, label, version = self.samples[idx]
        mri_vol = np.array(np.load(f"{self.mri_cache_dir}/{mri_key}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        pet_vol = np.array(np.load(f"{self.pet_cache_dir}/{pet_key}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        return torch.from_numpy(mri_vol).unsqueeze(0), torch.from_numpy(pet_vol).unsqueeze(0), torch.tensor(label, dtype=torch.long), mri_key

In [7]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for vol, labels, _ in loader:
        vol, labels = vol.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(vol), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for vol, labels, _ in loader:
            vol, labels = vol.to(device), labels.to(device)
            out = model(vol)
            total_loss += criterion(out, labels).item()
            preds_all.extend(out.argmax(1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    acc = np.mean(np.array(preds_all) == np.array(labels_all))
    tpr = recall_score(labels_all, preds_all, zero_division=0)
    tnr = specificity_score(labels_all, preds_all)
    return total_loss / len(loader), acc, tpr, tnr

def train_epoch_mm(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for mri_vol, pet_vol, labels, _ in loader:
        mri_vol, pet_vol, labels = mri_vol.to(device), pet_vol.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(mri_vol, pet_vol), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_mm(model, loader, criterion, device):
    model.eval()
    total_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for mri_vol, pet_vol, labels, _ in loader:
            mri_vol, pet_vol, labels = mri_vol.to(device), pet_vol.to(device), labels.to(device)
            out = model(mri_vol, pet_vol)
            total_loss += criterion(out, labels).item()
            preds_all.extend(out.argmax(1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    acc = np.mean(np.array(preds_all) == np.array(labels_all))
    tpr = recall_score(labels_all, preds_all, zero_division=0)
    tnr = specificity_score(labels_all, preds_all)
    return total_loss / len(loader), acc, tpr, tnr

In [8]:
def measure_inference_time(model, loader, device, is_multimodal, n_batches=20):
    model.eval()
    times = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches: break
            if is_multimodal:
                mri_vol, pet_vol, labels, _ = batch
                mri_vol, pet_vol = mri_vol.to(device), pet_vol.to(device)
                bs = mri_vol.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time()
                _ = model(mri_vol, pet_vol)
            else:
                vol, labels, _ = batch
                vol = vol.to(device)
                bs = vol.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time()
                _ = model(vol)
            if device.type == 'cuda': torch.cuda.synchronize()
            times.append((time.time() - t0) / bs)
    return np.mean(times), np.std(times)

def try_compute_flops(model, loader, device, is_multimodal):
    try:
        model.eval()
        batch = next(iter(loader))
        with torch.no_grad():
            if is_multimodal:
                mri_vol, pet_vol, labels, _ = batch
                inputs = (mri_vol[:1].to(device), pet_vol[:1].to(device))
            else:
                vol, labels, _ = batch
                inputs = (vol[:1].to(device),)
            macs, _ = profile(model, inputs=inputs, verbose=False)
        return macs * 2
    except Exception as e:
        print(f"  (FLOPs failed: {e})")
        return None

def run_one_seed(seed, model_class, train_loader, val_loader, test_loader, is_multimodal, save_prefix, max_epochs=101, patience=15):
    torch.manual_seed(seed); torch.cuda.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    model = model_class(brain_size=BRAIN_SIZE, d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    train_fn = train_epoch_mm if is_multimodal else train_epoch
    eval_fn = evaluate_mm if is_multimodal else evaluate

    best_val_loss, no_improve, best_epoch, total_time = float("inf"), 0, 0, 0
    save_path = f"{CKPT_DIR}/{save_prefix}_seed{seed}.pt"

    print(f"\n--- Seed {seed} ---")
    print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val Acc':>8} | {'Val TPR':>8} | {'Val TNR':>8} | {'Time':>6}")
    print("-" * 70)

    for epoch in range(1, max_epochs):
        t0 = time.time()
        train_loss = train_fn(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, val_tpr, val_tnr = eval_fn(model, val_loader, criterion, device)
        scheduler.step(val_loss)
        epoch_time = time.time() - t0
        total_time += epoch_time
        print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | {val_acc:>8.4f} | {val_tpr:>8.4f} | {val_tnr:>8.4f} | {epoch_time:>5.1f}s")

        if val_loss < best_val_loss:
            best_val_loss, best_epoch, no_improve = val_loss, epoch, 0
            torch.save(model.state_dict(), save_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch}. Best: {best_epoch}")
                break

    model.load_state_dict(torch.load(save_path, weights_only=True))
    test_loss, test_acc, test_tpr, test_tnr = eval_fn(model, test_loader, criterion, device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    inf_mean, inf_std = measure_inference_time(model, test_loader, device, is_multimodal)
    flops = try_compute_flops(model, test_loader, device, is_multimodal)

    print(f"\n  >>> Seed {seed} TEST: Acc={test_acc*100:.1f}% | TPR={test_tpr*100:.1f}% | TNR={test_tnr*100:.1f}% | "
          f"train_time={total_time/60:.1f}min | inf={inf_mean*1000:.2f}ms | {f'{flops/1e9:.2f}GFLOPs' if flops else 'N/A'}")

    return {"seed": seed, "acc": test_acc, "tpr": test_tpr, "tnr": test_tnr, "best_epoch": best_epoch,
            "train_time_sec": total_time, "n_params": n_params, "inf_time_ms": inf_mean * 1000, "flops": flops}

In [9]:
BATCH_SIZE = 1  
wb_mri_train = DataLoader(WholeBrainDataset(X_train, y_train, WB_MRI_CACHE_AUG, True, True), batch_size=BATCH_SIZE, shuffle=True)
wb_mri_val   = DataLoader(WholeBrainDataset(X_val, y_val, WB_MRI_CACHE_AUG, True, False), batch_size=BATCH_SIZE, shuffle=False)
wb_mri_test  = DataLoader(WholeBrainDataset(X_test, y_test, WB_MRI_CACHE_AUG, True, False), batch_size=BATCH_SIZE, shuffle=False)

print("=== WHOLE-BRAIN NATIVE MRI-ONLY: seed 1 ===")
wb_native_mri_results = [run_one_seed(1, WholeBrainModel, wb_mri_train, wb_mri_val, wb_mri_test, False, "vim_wholebrain_native_mri")]

=== WHOLE-BRAIN NATIVE MRI-ONLY: seed 1 ===

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.7052 |     0.6945 |   0.5000 |   1.0000 |   0.0000 | 458.2s
     2 |     0.6969 |     0.6945 |   0.5000 |   1.0000 |   0.0000 | 414.4s
     3 |     0.6913 |     0.6950 |   0.5000 |   1.0000 |   0.0000 | 411.7s
     4 |     0.6925 |     0.6989 |   0.5000 |   1.0000 |   0.0000 | 411.1s
     5 |     0.6907 |     0.6950 |   0.5000 |   1.0000 |   0.0000 | 412.2s
     6 |     0.6916 |     0.6957 |   0.5000 |   1.0000 |   0.0000 | 414.2s
     7 |     0.6863 |     0.6984 |   0.5000 |   1.0000 |   0.0000 | 408.6s
     8 |     0.6871 |     0.7036 |   0.5000 |   1.0000 |   0.0000 | 411.1s
     9 |     0.6887 |     0.6960 |   0.5000 |   1.0000 |   0.0000 | 411.8s
    10 |     0.6856 |     0.6936 |   0.5000 |   1.0000 |   0.0000 | 412.3s
    11 |     0.6839 |     0.6982 |   0.5000 

In [10]:
wb_pet_train = DataLoader(WholeBrainDataset(X_train, y_train, WB_PET_CACHE_AUG, False, True), batch_size=BATCH_SIZE, shuffle=True)
wb_pet_val   = DataLoader(WholeBrainDataset(X_val, y_val, WB_PET_CACHE_AUG, False, False), batch_size=BATCH_SIZE, shuffle=False)
wb_pet_test  = DataLoader(WholeBrainDataset(X_test, y_test, WB_PET_CACHE_AUG, False, False), batch_size=BATCH_SIZE, shuffle=False)

print("=== WHOLE-BRAIN NATIVE PET-ONLY: seed 1 ===")
wb_native_pet_results = [run_one_seed(1, WholeBrainModel, wb_pet_train, wb_pet_val, wb_pet_test, False, "vim_wholebrain_native_pet")]

=== WHOLE-BRAIN NATIVE PET-ONLY: seed 1 ===

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.7121 |     0.6924 |   0.5000 |   1.0000 |   0.0000 | 424.5s
     2 |     0.6988 |     0.6905 |   0.5000 |   1.0000 |   0.0000 | 412.1s
     3 |     0.6911 |     0.6887 |   0.5000 |   1.0000 |   0.0000 | 413.7s
     4 |     0.6909 |     0.6943 |   0.5000 |   1.0000 |   0.0000 | 412.6s
     5 |     0.6913 |     0.6806 |   0.6190 |   0.5238 |   0.7143 | 414.4s
     6 |     0.6903 |     0.6781 |   0.6190 |   0.5238 |   0.7143 | 414.2s
     7 |     0.6842 |     0.6847 |   0.5000 |   1.0000 |   0.0000 | 412.3s
     8 |     0.6780 |     0.6954 |   0.5000 |   1.0000 |   0.0000 | 412.8s
     9 |     0.6809 |     0.6651 |   0.6190 |   0.5238 |   0.7143 | 412.7s
    10 |     0.6743 |     0.6619 |   0.6667 |   0.5238 |   0.8095 | 415.0s
    11 |     0.6680 |     0.6592 |   0.6190 

In [11]:
wb_mm_train = DataLoader(MultimodalWholeBrainDataset(X_train, y_train, WB_MRI_CACHE_AUG, WB_PET_CACHE_AUG, True), batch_size=BATCH_SIZE, shuffle=True)
wb_mm_val   = DataLoader(MultimodalWholeBrainDataset(X_val, y_val, WB_MRI_CACHE_AUG, WB_PET_CACHE_AUG, False), batch_size=BATCH_SIZE, shuffle=False)
wb_mm_test  = DataLoader(MultimodalWholeBrainDataset(X_test, y_test, WB_MRI_CACHE_AUG, WB_PET_CACHE_AUG, False), batch_size=BATCH_SIZE, shuffle=False)

print("=== WHOLE-BRAIN NATIVE MULTIMODAL: seed 1 ===")
wb_native_mm_results = [run_one_seed(1, MultimodalWholeBrainModel, wb_mm_train, wb_mm_val, wb_mm_test, True, "vim_wholebrain_native_mm")]

=== WHOLE-BRAIN NATIVE MULTIMODAL: seed 1 ===

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.7066 |     0.6973 |   0.5000 |   0.0000 |   1.0000 | 1641.4s
     2 |     0.6985 |     0.6908 |   0.4762 |   0.9524 |   0.0000 | 1654.6s
     3 |     0.6936 |     0.6922 |   0.5000 |   1.0000 |   0.0000 | 1681.2s
     4 |     0.6983 |     0.6890 |   0.5714 |   0.4286 |   0.7143 | 1692.2s
     5 |     0.6865 |     0.6877 |   0.5000 |   1.0000 |   0.0000 | 1690.6s
     6 |     0.6844 |     0.6837 |   0.5952 |   0.9524 |   0.2381 | 1700.1s
     7 |     0.6821 |     0.6827 |   0.5238 |   0.9524 |   0.0952 | 1685.1s
     8 |     0.6689 |     0.6908 |   0.5000 |   0.0000 |   1.0000 | 1735.9s
     9 |     0.6721 |     0.7126 |   0.5000 |   1.0000 |   0.0000 | 1645.3s
    10 |     0.6703 |     0.6591 |   0.6429 |   0.5714 |   0.7143 | 1583.3s
    11 |     0.6602 |     0.6571

In [12]:
def summarize(results, name):
    r = results[0]
    print(f"{name}: Acc={r['acc']*100:.1f}% | TPR={r['tpr']*100:.1f}% | TNR={r['tnr']*100:.1f}% | "
          f"Params={r['n_params']:,} | Train={r['train_time_sec']/60:.1f}min | Inf={r['inf_time_ms']:.2f}ms")

print("=== Native resolution (no token-matching downsample) ===")
summarize(wb_native_mri_results, "MRI-only")
summarize(wb_native_pet_results, "PET-only")
summarize(wb_native_mm_results, "Multimodal")

print("\n=== Compare to token-matched whole-brain (downsampled to 3,072 tokens) ===")
print("MRI-only:   Acc=56.3±2.7% | TPR=65.1±7.3% | TNR=47.6±12.6%")
print("PET-only:   Acc=63.5±1.4% | TPR=52.4±0.0% | TNR=74.6±2.7%")
print("Multimodal: Acc=63.5±1.4% | TPR=55.6±2.7% | TNR=71.4±0.0%")

=== Native resolution (no token-matching downsample) ===
MRI-only: Acc=61.9% | TPR=42.9% | TNR=81.0% | Params=47,074 | Train=525.9min | Inf=60.12ms
PET-only: Acc=64.3% | TPR=57.1% | TNR=71.4% | Params=47,074 | Train=688.6min | Inf=60.50ms
Multimodal: Acc=69.0% | TPR=61.9% | TNR=76.2% | Params=94,146 | Train=2362.6min | Inf=78.92ms

=== Compare to token-matched whole-brain (downsampled to 3,072 tokens) ===
MRI-only:   Acc=56.3±2.7% | TPR=65.1±7.3% | TNR=47.6±12.6%
PET-only:   Acc=63.5±1.4% | TPR=52.4±0.0% | TNR=74.6±2.7%
Multimodal: Acc=63.5±1.4% | TPR=55.6±2.7% | TNR=71.4±0.0%


In [13]:
BATCH_SIZE = 1

# MRI-only: seeds 7, 123 (seed 1 already done)
print("=== WHOLE-BRAIN NATIVE MRI-ONLY: seeds 7, 123 ===")
wb_mri_native_remaining = [
    run_one_seed(s, WholeBrainModel, wb_mri_train, wb_mri_val, wb_mri_test, False, "vim_wholebrain_native_mri")
    for s in [7, 123]
]

=== WHOLE-BRAIN NATIVE MRI-ONLY: seeds 7, 123 ===

--- Seed 7 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.6999 |     0.6972 |   0.5000 |   1.0000 |   0.0000 | 412.5s
     2 |     0.7035 |     0.6954 |   0.5000 |   1.0000 |   0.0000 | 405.5s
     3 |     0.6952 |     0.6954 |   0.5000 |   1.0000 |   0.0000 | 399.3s
     4 |     0.6932 |     0.6940 |   0.5000 |   1.0000 |   0.0000 | 402.3s
     5 |     0.6906 |     0.6935 |   0.5000 |   1.0000 |   0.0000 | 406.2s
     6 |     0.6889 |     0.6949 |   0.5000 |   1.0000 |   0.0000 | 401.3s
     7 |     0.6882 |     0.6949 |   0.5000 |   1.0000 |   0.0000 | 403.2s
     8 |     0.6877 |     0.6961 |   0.5000 |   1.0000 |   0.0000 | 400.0s
     9 |     0.6859 |     0.6932 |   0.5000 |   1.0000 |   0.0000 | 403.3s
    10 |     0.6864 |     0.6966 |   0.5000 |   1.0000 |   0.0000 | 405.7s
    11 |     0.6853 |     0.6970 |   0

In [15]:
# PET-only: seeds 7, 123
print("=== WHOLE-BRAIN NATIVE PET-ONLY: seeds 7, 123 ===")
wb_pet_native_remaining = [
    run_one_seed(s, WholeBrainModel, wb_pet_train, wb_pet_val, wb_pet_test, False, "vim_wholebrain_native_pet")
    for s in [7, 123]
]

=== WHOLE-BRAIN NATIVE PET-ONLY: seeds 7, 123 ===

--- Seed 7 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.7044 |     0.6908 |   0.5000 |   1.0000 |   0.0000 | 406.5s
     2 |     0.7054 |     0.6890 |   0.5000 |   1.0000 |   0.0000 | 413.7s
     3 |     0.6941 |     0.6872 |   0.5000 |   1.0000 |   0.0000 | 410.2s
     4 |     0.6942 |     0.6880 |   0.5000 |   1.0000 |   0.0000 | 458.6s
     5 |     0.6909 |     0.6831 |   0.6429 |   0.5238 |   0.7619 | 414.5s
     6 |     0.6886 |     0.6839 |   0.5238 |   1.0000 |   0.0476 | 414.2s
     7 |     0.6866 |     0.6806 |   0.5476 |   0.9524 |   0.1429 | 414.4s
     8 |     0.6806 |     0.6766 |   0.5952 |   0.9524 |   0.2381 | 418.1s
     9 |     0.6766 |     0.6619 |   0.6429 |   0.5238 |   0.7619 | 416.2s
    10 |     0.6744 |     0.6566 |   0.6429 |   0.5714 |   0.7143 | 417.9s
    11 |     0.6667 |     0.6573 |   0

In [17]:
# Multimodal: seeds 7, 123
wb_mm_train = DataLoader(MultimodalWholeBrainDataset(X_train, y_train, WB_MRI_CACHE_AUG, WB_PET_CACHE_AUG, True), batch_size=BATCH_SIZE, shuffle=True)
wb_mm_val   = DataLoader(MultimodalWholeBrainDataset(X_val, y_val, WB_MRI_CACHE_AUG, WB_PET_CACHE_AUG, False), batch_size=BATCH_SIZE, shuffle=False)
wb_mm_test  = DataLoader(MultimodalWholeBrainDataset(X_test, y_test, WB_MRI_CACHE_AUG, WB_PET_CACHE_AUG, False), batch_size=BATCH_SIZE, shuffle=False)

print("=== WHOLE-BRAIN NATIVE MULTIMODAL: seeds 7, 123 ===")
wb_mm_native_remaining = [
    run_one_seed(s, MultimodalWholeBrainModel, wb_mm_train, wb_mm_val, wb_mm_test, True, "vim_wholebrain_native_mm")
    for s in [7, 123]
    ]

=== WHOLE-BRAIN NATIVE MULTIMODAL: seeds 7, 123 ===

--- Seed 7 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.7042 |     0.6914 |   0.5714 |   0.7619 |   0.3810 | 861.5s
     2 |     0.7008 |     0.7107 |   0.5000 |   1.0000 |   0.0000 | 808.3s
     3 |     0.6983 |     0.6981 |   0.5000 |   1.0000 |   0.0000 | 807.3s
     4 |     0.6857 |     0.7041 |   0.5000 |   1.0000 |   0.0000 | 806.2s
     5 |     0.6899 |     0.6908 |   0.5000 |   1.0000 |   0.0000 | 806.9s
     6 |     0.6886 |     0.6912 |   0.5000 |   1.0000 |   0.0000 | 808.3s
     7 |     0.6813 |     0.7004 |   0.5000 |   1.0000 |   0.0000 | 807.4s
     8 |     0.6841 |     0.6845 |   0.4762 |   0.9524 |   0.0000 | 807.3s
     9 |     0.6804 |     0.6772 |   0.6429 |   0.6190 |   0.6667 | 806.1s
    10 |     0.6748 |     0.6712 |   0.6429 |   0.5714 |   0.7143 | 807.6s
    11 |     0.6655 |     0.6790 |  

In [18]:
seed1_mri_result = {"seed": 1, "acc": 0.619, "tpr": 0.429, "tnr": 0.810}
seed1_pet_result = {"seed": 1, "acc": 0.643, "tpr": 0.571, "tnr": 0.714}
seed1_mm_result  = {"seed": 1, "acc": 0.690, "tpr": 0.619, "tnr": 0.762}

wb_mri_native_full = [seed1_mri_result] + wb_mri_native_remaining
wb_pet_native_full = [seed1_pet_result] + wb_pet_native_remaining
wb_mm_native_full  = [seed1_mm_result] + wb_mm_native_remaining

def summarize(results, name):
    accs = [r["acc"] for r in results]
    tprs = [r["tpr"] for r in results]
    tnrs = [r["tnr"] for r in results]
    print(f"{name}: Acc={np.mean(accs)*100:.1f}±{np.std(accs,ddof=1)*100:.1f}% | "
          f"TPR={np.mean(tprs)*100:.1f}±{np.std(tprs,ddof=1)*100:.1f}% | "
          f"TNR={np.mean(tnrs)*100:.1f}±{np.std(tnrs,ddof=1)*100:.1f}%")

summarize(wb_mri_native_full, "MRI-only, native (3-seed)")
summarize(wb_pet_native_full, "PET-only, native (3-seed)")
summarize(wb_mm_native_full, "Multimodal, native (3-seed)")

print("\n=== Compare to token-matched whole-brain (downsampled to 3,072 tokens) ===")
print("MRI-only:   Acc=56.3±2.7% | TPR=65.1±7.3% | TNR=47.6±12.6%")
print("PET-only:   Acc=63.5±1.4% | TPR=52.4±0.0% | TNR=74.6±2.7%")
print("Multimodal: Acc=63.5±1.4% | TPR=55.6±2.7% | TNR=71.4±0.0%")

MRI-only, native (3-seed): Acc=57.9±6.9% | TPR=68.3±29.1% | TNR=47.6±42.3%
PET-only, native (3-seed): Acc=64.3±0.0% | TPR=57.1±0.0% | TNR=71.4±0.0%
Multimodal, native (3-seed): Acc=69.0±2.4% | TPR=58.7±2.7% | TNR=79.4±5.5%

=== Compare to token-matched whole-brain (downsampled to 3,072 tokens) ===
MRI-only:   Acc=56.3±2.7% | TPR=65.1±7.3% | TNR=47.6±12.6%
PET-only:   Acc=63.5±1.4% | TPR=52.4±0.0% | TNR=74.6±2.7%
Multimodal: Acc=63.5±1.4% | TPR=55.6±2.7% | TNR=71.4±0.0%


In [2]:
# To Investigate lack of variance in PET-only, natvie:

import os, torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from mambapy.vim import VMamba, MambaConfig

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class VimEncoder(nn.Module):
    def __init__(self, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        config = MambaConfig(d_model=d_model, n_layers=n_layers, d_state=d_state,
                              bidirectional=True, divide_output=True, pscan=True, use_cuda=False)
        self.encoder = VMamba(config)
        self.final_norm = nn.LayerNorm(d_model)
    def forward(self, tokens):
        return self.final_norm(self.encoder(tokens))

class WholeBrainPatchEmbed3D(nn.Module):
    def __init__(self, brain_size=256, patch_size=8, d_model=32):
        super().__init__()
        self.patch_size = patch_size
        self.grid_size = brain_size // patch_size
        self.n_tokens = self.grid_size ** 3
        self.d_model = d_model
        self.patch_conv = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.depth_embed = nn.Embedding(self.grid_size, d_model)
        self.height_embed = nn.Embedding(self.grid_size, d_model)
        self.width_embed = nn.Embedding(self.grid_size, d_model)
        with torch.no_grad():
            for emb in [self.depth_embed, self.height_embed, self.width_embed]:
                emb.weight.mul_(0.02)
        d, h, w = torch.meshgrid(torch.arange(self.grid_size), torch.arange(self.grid_size),
                                  torch.arange(self.grid_size), indexing="ij")
        self.register_buffer("coordinates", torch.stack([d, h, w], dim=-1).reshape(-1, 3), persistent=False)

    def forward(self, volume):
        tokens = self.patch_conv(volume).flatten(2).transpose(1, 2)
        coords = self.coordinates
        spatial = self.depth_embed(coords[:, 0]) + self.height_embed(coords[:, 1]) + self.width_embed(coords[:, 2])
        tokens = tokens + spatial[None, :, :]
        occupancy = F.max_pool3d((volume.abs() > 1e-6).float(), kernel_size=self.patch_size, stride=self.patch_size)
        valid = occupancy.flatten(1).bool()
        tokens = tokens * valid.unsqueeze(-1).to(tokens.dtype)
        return tokens, valid

class WholeBrainBranch(nn.Module):
    def __init__(self, brain_size=256, patch_size=8, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        self.patch_embed = WholeBrainPatchEmbed3D(brain_size, patch_size, d_model)
        self.vim = VimEncoder(d_model, n_layers, d_state)
    def forward(self, volume):
        tokens, valid = self.patch_embed(volume)
        tokens = self.vim(tokens)
        w = valid.unsqueeze(-1).to(tokens.dtype)
        pooled = (tokens * w).sum(dim=1) / w.sum(dim=1).clamp_min(1.0)
        return pooled

class WholeBrainModel(nn.Module):
    def __init__(self, brain_size=256, patch_size=8, d_model=32, n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.branch = WholeBrainBranch(brain_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)
    def forward(self, volume):
        pooled = self.branch(volume)
        return self.classifier(self.dropout(pooled))

class WholeBrainDataset(Dataset):
    def __init__(self, sessions, labels, cache_dir, is_mri=True, is_train=False):
        self.samples, self.cache_dir = [], cache_dir
        for session_id, label in zip(sessions, labels):
            key = session_id if is_mri else session_to_subject[session_id]
            self.samples.append((key, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((key, label, f"aug{seed}"))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        key, label, version = self.samples[idx]
        vol = np.array(np.load(f"{self.cache_dir}/{key}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        return torch.from_numpy(vol).unsqueeze(0), torch.tensor(label, dtype=torch.long), key

COHORT_CSV       = "D:/mamba_model/thesis_cohort_final.csv"
WB_PET_CACHE_AUG = "D:/mamba_model/preprocessed_cache_wholebrain_native_pet_aug"
CKPT_DIR         = "D:/mamba_model/checkpoints"

df = pd.read_csv(COHORT_CSV)
sessions, labels = df["mri_session"].values, df["outcome_label"].values
X_tv, X_test, y_tv, y_test = train_test_split(sessions, labels, test_size=0.2, random_state=42, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv)
session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))

wb_pet_test = DataLoader(WholeBrainDataset(X_test, y_test, WB_PET_CACHE_AUG, False, False), batch_size=1, shuffle=False)

# Now the actual comparison
for seed in [1, 7, 123]:
    model = WholeBrainModel(brain_size=256, d_model=32, n_layers=2).to(device)
    model.load_state_dict(torch.load(f"{CKPT_DIR}/vim_wholebrain_native_pet_seed{seed}.pt", weights_only=True))
    model.eval()
    preds = []
    with torch.no_grad():
        for vol, lbl, key in wb_pet_test:
            vol = vol.to(device)
            preds.extend(model(vol).argmax(1).cpu().numpy())
    print(f"Seed {seed} predictions: {preds}")

Seed 1 predictions: [np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1)]
Seed 7 predictions: [np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(